In [ ]:
import base64
import hashlib
import json
import re
import types
from pathlib import Path
import numpy as np
import pandas as pd

sf_options = {"sfUrl": "", "sfUser": "", "sfPassword": "", "sfRole": "", "sfWarehouse": "", "sfDatabase": "", "sfSchema": ""}
LOCAL_REPORT_PATHS = []
REMOTE_REPORT_TABLES = []
OUTPUT_DIRECTORY = ""
MAX_REPORTS = 100
print("Aggregate experiment tracker: no training, prediction arrays or raw-claim reads.")
print("Historic results remain reported evidence. Missing outcomes remain unavailable.")


In [ ]:
TRACKER_SOURCE = '"""Bounded aggregate evidence ledger; never trains or reads prediction arrays."""\nimport hashlib\nimport json\nimport math\nimport re\nfrom pathlib import Path\n\nimport pandas as pd\n\n\nMAX_REPORT_BYTES = 8 * 1024 * 1024\nDISPLAY_COLUMNS = ["Experiment", "What changed", "Population/split", "Lift@10%", "Matched delta@10%",\n                   "Additional captures@10%", "Conclusion", "Remaining limitation", "Evidence", "Status", "Source"]\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()\n\n\ndef number(value):\n    if value is None:\n        return None\n    if isinstance(value, bool) or not math.isfinite(float(value)):\n        raise ValueError("A metric must be finite or explicitly missing")\n    return float(value)\n\n\ndef _json(blob):\n    if len(blob) > MAX_REPORT_BYTES:\n        raise ValueError("Aggregate report exceeds the configured byte bound")\n    result = json.loads(blob.decode("utf-8"), parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))\n    if not isinstance(result, dict):\n        raise ValueError("An aggregate report must be an object")\n    return result\n\n\ndef load_local_report(path):\n    path = Path(path)\n    if path.is_dir():\n        path = path / "report.json"\n    if path.stat().st_size > MAX_REPORT_BYTES:\n        raise ValueError("Aggregate report exceeds the configured byte bound")\n    blob = path.read_bytes()\n    receipt_path = path.with_name("report_receipt.json")\n    if receipt_path.stat().st_size > 256 * 1024:\n        raise ValueError("Oversized report receipt")\n    receipt = _json(receipt_path.read_bytes())\n    expected = receipt.get("sha256")\n    if isinstance(expected, dict):\n        expected = expected.get("report.json")\n    if expected is None:\n        expected = receipt.get("hashes", {}).get("report.json")\n    actual = hashlib.sha256(blob).hexdigest()\n    if expected != actual:\n        raise ValueError("Aggregate report receipt does not match; retry after the active writer commits")\n    return _json(blob), {"source": str(path), "sha256": actual, "integrity_verified": True}\n\n\ndef load_remote_report(read_query, table, decoder):\n    """Read only the bounded report.json artifact with the existing checked decoder."""\n    if not isinstance(table, str) or not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9$]*\\.[A-Za-z_][A-Za-z_0-9$]*\\.[A-Za-z_][A-Za-z_0-9$]*", table):\n        raise ValueError("Use one explicit fully qualified artifact table")\n    quoted = ".".join(\'"\' + part.upper() + \'"\' for part in table.split("."))\n    inventory = read_query(f"SELECT MIN(BYTE_LENGTH) AS MIN_BYTES, MAX(BYTE_LENGTH) AS MAX_BYTES, "\n                           f"MAX(CHUNK_COUNT) AS MAX_CHUNKS, COUNT(*) AS STORED_CHUNKS, "\n                           f"SUM(LENGTH(PAYLOAD_BASE64)) AS ENCODED_BYTES FROM {quoted} "\n                           "WHERE ARTIFACT_NAME = \'report.json\'", 1)\n    if len(inventory) != 1:\n        raise ValueError("Missing aggregate report inventory")\n    item = inventory.iloc[0]\n    if not (0 <= int(item.MIN_BYTES) <= int(item.MAX_BYTES) <= MAX_REPORT_BYTES):\n        raise ValueError("Missing or oversized report")\n    chunks = int(item.MAX_CHUNKS)\n    if not 1 <= chunks <= 10000 or int(item.STORED_CHUNKS) != chunks:\n        raise ValueError("Invalid report chunk inventory")\n    if int(item.ENCODED_BYTES) > 4 * ((MAX_REPORT_BYTES + 2) // 3):\n        raise ValueError("Oversized encoded report")\n    cols = "CHUNK_INDEX,CHUNK_COUNT,BYTE_LENGTH,SHA256,PAYLOAD_BASE64"\n    frame = read_query(f"SELECT {cols} FROM {quoted} WHERE ARTIFACT_NAME = \'report.json\' ORDER BY CHUNK_INDEX", chunks)\n    blob = decoder(frame, MAX_REPORT_BYTES)\n    return _json(blob), {"source": table + "/report.json", "sha256": hashlib.sha256(blob).hexdigest(),\n                        "integrity_verified": True}\n\n\ndef interval_conclusion(delta, lower, upper):\n    if delta is None:\n        return "No matched result available"\n    if lower is None or upper is None:\n        return "Observed difference only; paired uncertainty unavailable"\n    if lower > upper or not lower <= delta <= upper:\n        # A percentile bootstrap need not contain the point estimate.\n        if lower > upper:\n            raise ValueError("Reversed interval")\n    if lower > 0:\n        return "Higher observed development lift; interval excludes zero"\n    if upper < 0:\n        return "Lower observed development lift; interval excludes zero"\n    return "Ordering unresolved; interval includes zero (not equivalence)"\n\n\ndef _row(experiment, changed, population, lift, source, **extra):\n    row = {"Experiment": experiment, "What changed": changed, "Population/split": population,\n           "Lift@10%": number(lift), "Matched delta@10%": None, "Additional captures@10%": None,\n           "Conclusion": "Result reported; no matched contrast established", "Remaining limitation": "",\n           "Evidence": "SAVED_AGGREGATE_INTEGRITY_VERIFIED", "Status": "COMPLETED_REPORTED",\n           "Source": source, "comparison_block": None, "split": None, "metric": "Lift@10%",\n           "lower": None, "upper": None, "train_snapshots": None, "validation_identity": None,\n           "protocol_identity": None, "version": None, "arm": None}\n    row.update(extra)\n    return row\n\n\ndef pilot_rows(report, receipt):\n    if report.get("schema") != "cached_pilot_report_v1" or not receipt.get("integrity_verified"):\n        raise ValueError("A checked pilot report is required")\n    identity = report["identity"]\n    for name in ("evaluation_keys_sha256", "evaluation_labels_sha256", "preparation_hash"):\n        if not re.fullmatch("[0-9a-f]{64}", str(identity.get(name, ""))):\n            raise ValueError("Missing pilot evaluation/preparation identity: " + name)\n    evaluation = digest({k: identity[k] for k in ("evaluation_keys_sha256", "evaluation_labels_sha256")})\n    definition_hash = identity.get("feature_definition_hash")\n    implementation_protocol = identity.get("protocol_hash")\n    protocol = (digest({"settings": identity["settings"], "feature_names": identity["feature_names"],\n                        "feature_definition_hash": definition_hash, "protocol_hash": implementation_protocol})\n                if all(re.fullmatch("[0-9a-f]{64}", str(value)) for value in (definition_hash, implementation_protocol)) else None)\n    grouped = {}\n    for entry in report["rows"]:\n        split = str(entry["Split"]).upper()\n        if split not in {"TRAIN", "VALIDATION"}:\n            raise ValueError("Pilot reports cannot expose TEST")\n        key = (split, str(entry["Model"]))\n        if key in grouped:\n            raise ValueError("Duplicate pilot result row")\n        grouped[key] = entry\n    results = []\n    for (split, arm), entry in grouped.items():\n        n, positives = int(entry["Snapshots"]), int(entry["Positive_snapshots"])\n        if n <= 0 or not 0 < positives < n:\n            raise ValueError("Pilot result requires both classes")\n        lift = number(entry["Lift@10%"])\n        captured, selected = int(entry["Captured@10%"]), int(entry["Selected@10%"])\n        if selected != math.ceil(.1 * n) or not 0 <= captured <= min(positives, selected):\n            raise ValueError("Pilot top-decile counts are inconsistent")\n        if lift is None or not math.isclose(lift, captured / selected / (positives / n), rel_tol=1e-8, abs_tol=1e-8):\n            raise ValueError("Pilot lift does not match its aggregate counts")\n        row = _row("Pilot " + str(identity.get("version", "")) + " / " + arm, arm,\n                   f"{n:,} snapshots / {int(entry[\'Patients\']):,} patients / {positives:,} positives / {split}",\n                   lift, receipt["source"], split=split,\n                   comparison_block="pilot:" + receipt["sha256"] + ":" + split,\n                   validation_identity=evaluation if split == "VALIDATION" else None,\n                   protocol_identity=protocol, train_snapshots=int(identity["train_snapshots"]),\n                   version=identity.get("version"), arm=arm, artifact_sha256=receipt["sha256"],\n                   **{"Evidence": "SAVED_PILOT_AGGREGATE_INTEGRITY_VERIFIED",\n                      "Remaining limitation": "Convenience pilot; repeated development validation; no out-of-time claim. "\n                                              "Integrity checks do not establish historical source availability.",\n                      "Conclusion": "Baseline" if arm == "AUTOML" else "Matched point difference pending"})\n        baseline = grouped.get((split, "AUTOML"))\n        if baseline is not None:\n            if any(entry[k] != baseline[k] for k in ("Snapshots", "Patients", "Positive_snapshots", "Selected@10%")):\n                raise ValueError("Pilot arms have different evaluation populations")\n            row["Matched delta@10%"] = lift - float(baseline["Lift@10%"])\n            row["Additional captures@10%"] = captured - int(baseline["Captured@10%"])\n        pair = report.get("paired_lift", {}).get(arm, {}) if split == "VALIDATION" else {}\n        interval = pair.get("lift_by_fraction", {}).get("0.1", {})\n        if interval:\n            if pair.get("paired") is not True or pair.get("unit") != "patient_cluster":\n                raise ValueError("Expected paired patient-cluster uncertainty")\n            delta = number(interval["delta"])\n            if row["Matched delta@10%"] is None or not math.isclose(delta, row["Matched delta@10%"], abs_tol=1e-8):\n                raise ValueError("Paired interval direction differs from candidate minus baseline")\n            lower = number(interval.get("lower")) if pair.get("interval_reliable") else None\n            upper = number(interval.get("upper")) if pair.get("interval_reliable") else None\n            row.update(lower=lower, upper=upper)\n            row["Conclusion"] = interval_conclusion(delta, lower, upper)\n        elif arm != "AUTOML":\n            row["Conclusion"] = interval_conclusion(row["Matched delta@10%"], None, None)\n        results.append(row)\n    return results\n\n\ndef saved_rows(report, receipt):\n    if report.get("schema") == "cached_pilot_report_v1":\n        return pilot_rows(report, receipt)\n    if not receipt.get("integrity_verified"):\n        raise ValueError("Saved report integrity must be verified")\n    # Existing timing experiment reports are aggregate rows plus a paired-lift bundle.\n    if "rows" in report and "identity" in report and "scope" in report:\n        out = []\n        for entry in report["rows"]:\n            if not {"Model", "Lift@10%", "Snapshots", "Split"} <= set(entry):\n                raise ValueError("Unsupported saved result row; add an explicit schema adapter")\n            out.append(_row(str(entry["Model"]), "Saved aggregate feature comparison",\n                       f"{int(entry[\'Snapshots\']):,} snapshots / {entry[\'Split\']}", entry["Lift@10%"], receipt["source"],\n                       comparison_block=None, split=str(entry["Split"]).upper(),\n                       **{"Remaining limitation": str(report["scope"]),\n                          "Conclusion": "Saved reported result; comparison identity requires its protocol receipt"}))\n        return out\n    # Existing production comparison reports encode metric dictionaries, not arrays.\n    if {"metrics", "snapshots", "split", "evaluation_scope", "prediction_identity_sha256"} <= set(report):\n        out = []\n        for model, metric in report["metrics"].items():\n            top = {float(x["fraction"]): x["lift"] for x in metric.get("topk", [])}\n            if .1 not in top:\n                continue\n            out.append(_row(model, "Saved comparison on aligned prediction keys",\n                       f"{int(report[\'snapshots\']):,} snapshots / {str(report[\'split\']).upper()}", top[.1], receipt["source"],\n                       comparison_block="saved:" + receipt["sha256"], split=str(report["split"]).upper(),\n                       **{"Remaining limitation": "; ".join(report.get("limitations", [])),\n                          "Conclusion": report.get("benchmark", {}).get("statement", str(report["evaluation_scope"]))}))\n        return out\n    raise ValueError("Unsupported aggregate report schema; no result was inferred")\n\n\ndef ledger(seed, reports=()):\n    if seed.get("schema") != "experiment_evidence_seed_v1":\n        raise ValueError("Unsupported evidence seed")\n    rows = []\n    for entry in seed["experiments"]:\n        row = _row(entry["experiment"], entry["changed"], entry["population"], entry.get("lift10"), entry["source"],\n                   comparison_block=entry.get("comparison_block"), split=entry.get("split"),\n                   metric=entry.get("metric", "Lift@10%"),\n                   **{"Evidence": entry["evidence"], "Status": entry["status"], "Conclusion": entry["conclusion"],\n                      "Remaining limitation": entry["limitation"], "precision": entry.get("precision", "unavailable")})\n        if entry.get("delta10") is not None:\n            if not entry.get("comparison_block"):\n                raise ValueError("A contrast requires a declared comparison block")\n            row["Matched delta@10%"] = number(entry["delta10"])\n        row["lower"], row["upper"] = number(entry.get("lower")), number(entry.get("upper"))\n        rows.append(row)\n    seen = set()\n    errors = []\n    for report, receipt in reports:\n        sha = receipt.get("sha256")\n        if sha in seen:\n            continue\n        try:\n            rows.extend(saved_rows(report, receipt))\n            seen.add(sha)\n        except (KeyError, ValueError, TypeError) as exc:\n            errors.append({"Source": receipt.get("source"), "Status": "BLOCKED", "Reason": str(exc)})\n    return pd.DataFrame(rows), pd.DataFrame(seed.get("engineering", [])), pd.DataFrame(errors)\n\n\ndef chart_groups(frame):\n    groups = []\n    for key, group in frame.dropna(subset=["comparison_block", "Lift@10%"]).groupby(["comparison_block", "split", "metric"], dropna=False, sort=False):\n        if len(group) >= 2:\n            groups.append((key, group.copy()))\n    return groups\n\n\ndef progress_groups(frame):\n    selected = frame[(frame["split"] == "VALIDATION") & frame["validation_identity"].notna() & frame["protocol_identity"].notna()]\n    result = []\n    for key, group in selected.groupby(["validation_identity", "protocol_identity", "arm"], sort=False):\n        group = group.drop_duplicates(["version", "train_snapshots"]).sort_values("train_snapshots", kind="stable")\n        if len(group) >= 2:\n            result.append((key, group))\n    return result\n\n\ndef plot_matched(frame):\n    import matplotlib.pyplot as plt\n    figures = []\n    for (block, split, metric), group in chart_groups(frame):\n        fig, ax = plt.subplots(figsize=(10, max(2.6, len(group) * .48)))\n        labels = group["Experiment"].tolist()\n        ax.barh(labels, group["Lift@10%"], color="#007F86")\n        ax.invert_yaxis()\n        ax.set_xlabel(metric + " (higher is better)")\n        ax.set_title(str(block)[:75] + " | " + str(split), loc="left")\n        ax.spines[["top", "right"]].set_visible(False)\n        for i, value in enumerate(group["Lift@10%"]):\n            ax.text(value, i, f"  {value:.3f}", va="center", fontsize=9)\n        fig.tight_layout()\n        figures.append(fig)\n    return figures\n\n\ndef plot_progress(frame):\n    import matplotlib.pyplot as plt\n    figures = []\n    for (_, _, arm), group in progress_groups(frame):\n        fig, ax = plt.subplots(figsize=(8, 3.4))\n        ax.plot(group["train_snapshots"], group["Lift@10%"], marker="o", color="#007F86")\n        ax.set(xlabel="TRAIN snapshots", ylabel="Validation Lift@10%", title=arm + " | fixed validation panel and feature definitions")\n        ax.spines[["top", "right"]].set_visible(False)\n        fig.tight_layout()\n        figures.append(fig)\n    return figures\n'
DECODER_SOURCE = "def decode_checked_artifact(frame, maximum_bytes):\n    if frame.empty:\n        raise ValueError('No artifact chunks returned.')\n    rows = frame.sort_values('CHUNK_INDEX').to_dict('records')\n    def integer(v):\n        if isinstance(v,(bool,np.bool_)) or not np.isfinite(float(v)) or int(v) != float(v):\n            raise ValueError('Nonintegral chunk metadata.')\n        return int(v)\n    count=integer(rows[0]['CHUNK_COUNT']); size=integer(rows[0]['BYTE_LENGTH'])\n    digest=str(rows[0]['SHA256']).lower()\n    if not 1 <= count <= 10000 or not 0 <= size <= maximum_bytes or not re.fullmatch(r'[0-9a-f]{64}',digest):\n        raise ValueError('Invalid or oversized artifact metadata.')\n    if len(rows)!=count or [integer(r['CHUNK_INDEX']) for r in rows] != list(range(count)):\n        raise ValueError('Missing or duplicate artifact chunks.')\n    if any(integer(r['CHUNK_COUNT'])!=count or integer(r['BYTE_LENGTH'])!=size or str(r['SHA256']).lower()!=digest for r in rows):\n        raise ValueError('Inconsistent artifact metadata.')\n    pieces=[]; encoded_size=0\n    for row in rows:\n        piece=row['PAYLOAD_BASE64']\n        if not isinstance(piece,str):\n            raise ValueError('Non-text payload encoding.')\n        encoded_size += len(piece)\n        if encoded_size > 4*((maximum_bytes+2)//3):\n            raise ValueError('Encoded payload exceeds byte bound.')\n        pieces.append(piece)\n    blob=base64.b64decode(''.join(pieces),validate=True)\n    if len(blob)!=size or hashlib.sha256(blob).hexdigest()!=digest:\n        raise ValueError('Artifact length or SHA-256 verification failed.')\n    return blob"
EVIDENCE_SOURCE = '{\n  "schema": "experiment_evidence_seed_v1",\n  "as_of": "2026-11-06",\n  "source_registry_sha256": {\n    "experiment_registry.csv": "2043afbd194e85943a9781422d5da1ba7a99a4703e7c5cf43a4d3e0084100835",\n    "benchmark_registry.csv": "18a2100f1e2a2b7c2a4799eefcbc8c3e3e86084c8f7bf7d5a9c471191e251292"\n  },\n  "experiments": [\n    {\n      "experiment": "V63_EXISTING",\n      "changed": "49 V63; tabular",\n      "population": "original fitted/evaluation population unresolved",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 2",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "COUNT_RUN001",\n      "changed": "1028 monthly category counts; 12",\n      "population": "patient-snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 3",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "ITERATION_V2",\n      "changed": "all1028/reduced150/core; 12",\n      "population": "patient-snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 4",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "SELECTED_MANUAL49",\n      "changed": "configured49 V63; monthly/quarterly variants",\n      "population": "patient-snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 5",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "TEMPORAL_TOP50_BUILDER",\n      "changed": "1028 with temporal ranking/top50; 12",\n      "population": "patient-snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 6",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "SNAPSHOT_PAD_DELIVERY",\n      "changed": "49 V63 snapshot vectors repeated at valid positions; 12 monthly/4 quarterly",\n      "population": "patient-snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 9",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "RELIANCE49_PHOTOS",\n      "changed": "49 V63 snapshot features; monthly/quarterly",\n      "population": "permutation uses validation snapshots; correlations latest patient snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 10",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "FINAL49_REQUESTED",\n      "changed": "49 ranked; preserve agreed representation",\n      "population": "patient-snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 15",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "FINAL1028_REQUESTED",\n      "changed": "1028 ranked; 12",\n      "population": "patient-snapshot",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 16",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "RIGOROUS1028_V1_DELIVERY",\n      "changed": "1028 raw candidates; final count selected only by actual development runs; 12 saved monthly positions; declared representation ablations",\n      "population": "patient-snapshot; patient-disjoint frozen splits; latest-patient secondary",\n      "lift10": null,\n      "source": "project_context/experiments/experiment_registry.csv row 17",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_RESULT_NOT_RECOVERED",\n      "conclusion": "Implementation or exploratory source recorded; no lift inferred",\n      "limitation": "Source delivery does not establish a completed clinical fit"\n    },\n    {\n      "experiment": "CLEAN25_MONTHLY train",\n      "changed": "49 clean/hybrid; saved manifest pending",\n      "population": "patient-snapshot / train",\n      "lift10": 3.601664,\n      "source": "project_context/experiments/benchmark_registry.csv CLEAN25_MONTHLY train",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Historical reported result; exact population lineage unresolved",\n      "limitation": "Not grouped with later cohorts; saved preparation manifest unavailable",\n      "split": "TRAIN",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "CLEAN25_MONTHLY validation",\n      "changed": "49 clean/hybrid; saved manifest pending",\n      "population": "patient-snapshot / validation",\n      "lift10": 2.71575,\n      "source": "project_context/experiments/benchmark_registry.csv CLEAN25_MONTHLY validation",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Historical reported result; exact population lineage unresolved",\n      "limitation": "Not grouped with later cohorts; saved preparation manifest unavailable",\n      "split": "VALIDATION",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "CLEAN25_QUARTERLY train",\n      "changed": "49 clean/hybrid; saved manifest pending",\n      "population": "patient-snapshot / train",\n      "lift10": 3.452922,\n      "source": "project_context/experiments/benchmark_registry.csv CLEAN25_QUARTERLY train",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Historical reported result; exact population lineage unresolved",\n      "limitation": "Not grouped with later cohorts; saved preparation manifest unavailable",\n      "split": "TRAIN",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "CLEAN25_QUARTERLY validation",\n      "changed": "49 clean/hybrid; saved manifest pending",\n      "population": "patient-snapshot / validation",\n      "lift10": 2.616996,\n      "source": "project_context/experiments/benchmark_registry.csv CLEAN25_QUARTERLY validation",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Historical reported result; exact population lineage unresolved",\n      "limitation": "Not grouped with later cohorts; saved preparation manifest unavailable",\n      "split": "VALIDATION",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "SELECT30_all_eligible",\n      "changed": "all_eligible selector with fixed L2 logistic evaluator",\n      "population": "Latest TRAIN snapshot per patient / three TRAIN folds",\n      "lift10": 1.855781,\n      "source": "project_context/experiments/benchmark_registry.csv SELECT30_all_eligible Lift@10%_mean",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Feature selection improved the recorded mean fold lift versus all eligible columns",\n      "limitation": "Fold mean on a different population; not full-cohort Transformer performance",\n      "comparison_block": "SELECT30_TRAIN_CV",\n      "split": "TRAIN_CV",\n      "metric": "Mean fold Lift@10%",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "SELECT30_embedded",\n      "changed": "embedded selector with fixed L2 logistic evaluator",\n      "population": "Latest TRAIN snapshot per patient / three TRAIN folds",\n      "lift10": 2.194918,\n      "source": "project_context/experiments/benchmark_registry.csv SELECT30_embedded Lift@10%_mean",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Feature selection improved the recorded mean fold lift versus all eligible columns",\n      "limitation": "Fold mean on a different population; not full-cohort Transformer performance",\n      "comparison_block": "SELECT30_TRAIN_CV",\n      "split": "TRAIN_CV",\n      "metric": "Mean fold Lift@10%",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "SELECT30_filter",\n      "changed": "filter selector with fixed L2 logistic evaluator",\n      "population": "Latest TRAIN snapshot per patient / three TRAIN folds",\n      "lift10": 2.216343,\n      "source": "project_context/experiments/benchmark_registry.csv SELECT30_filter Lift@10%_mean",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Feature selection improved the recorded mean fold lift versus all eligible columns",\n      "limitation": "Fold mean on a different population; not full-cohort Transformer performance",\n      "comparison_block": "SELECT30_TRAIN_CV",\n      "split": "TRAIN_CV",\n      "metric": "Mean fold Lift@10%",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "SELECT30_wrapper",\n      "changed": "wrapper selector with fixed L2 logistic evaluator",\n      "population": "Latest TRAIN snapshot per patient / three TRAIN folds",\n      "lift10": 2.216377,\n      "source": "project_context/experiments/benchmark_registry.csv SELECT30_wrapper Lift@10%_mean",\n      "evidence": "DOCUMENTED_PHOTO_TRANSCRIPTION",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Feature selection improved the recorded mean fold lift versus all eligible columns",\n      "limitation": "Fold mean on a different population; not full-cohort Transformer performance",\n      "comparison_block": "SELECT30_TRAIN_CV",\n      "split": "TRAIN_CV",\n      "metric": "Mean fold Lift@10%",\n      "precision": "6 displayed decimals"\n    },\n    {\n      "experiment": "OLD_B / COUNT",\n      "changed": "Monthly count Transformer",\n      "population": "3,481 snapshots / 1,867 patients / 202 positive snapshots / VALIDATION",\n      "lift10": 2.073846,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus COUNT; temporal fusion ranked highest in this comparison",\n      "limitation": "Development validation; paired uncertainty unavailable; static control differs from the 49-feature refit",\n      "comparison_block": "R20261023_OLD_B_VALIDATION",\n      "split": "VALIDATION",\n      "precision": "6 displayed decimals",\n      "delta10": 0.0\n    },\n    {\n      "experiment": "OLD_B / STATIC_FUSION",\n      "changed": "Counts plus current AutoML summaries",\n      "population": "3,481 snapshots / 1,867 patients / 202 positive snapshots / VALIDATION",\n      "lift10": 2.518242,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus COUNT; temporal fusion ranked highest in this comparison",\n      "limitation": "Development validation; paired uncertainty unavailable; static control differs from the 49-feature refit",\n      "comparison_block": "R20261023_OLD_B_VALIDATION",\n      "split": "VALIDATION",\n      "precision": "6 displayed decimals",\n      "delta10": 0.444396\n    },\n    {\n      "experiment": "OLD_B / TEMPORAL_FUSION",\n      "changed": "Counts plus sequential AutoML",\n      "population": "3,481 snapshots / 1,867 patients / 202 positive snapshots / VALIDATION",\n      "lift10": 2.715751,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus COUNT; temporal fusion ranked highest in this comparison",\n      "limitation": "Development validation; paired uncertainty unavailable; static control differs from the 49-feature refit",\n      "comparison_block": "R20261023_OLD_B_VALIDATION",\n      "split": "VALIDATION",\n      "precision": "6 displayed decimals",\n      "delta10": 0.641905\n    },\n    {\n      "experiment": "OLD_B / LIGHTGBM_TRAIN_ONLY_STATIC",\n      "changed": "TRAIN-selected static AutoML plus availability",\n      "population": "3,481 snapshots / 1,867 patients / 202 positive snapshots / VALIDATION",\n      "lift10": 2.567619,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus COUNT; temporal fusion ranked highest in this comparison",\n      "limitation": "Development validation; paired uncertainty unavailable; static control differs from the 49-feature refit",\n      "comparison_block": "R20261023_OLD_B_VALIDATION",\n      "split": "VALIDATION",\n      "precision": "6 displayed decimals",\n      "delta10": 0.493773\n    },\n    {\n      "experiment": "ADDENDUM / V0",\n      "changed": "Locked count Transformer",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 2.866,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V0",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": 0.0\n    },\n    {\n      "experiment": "ADDENDUM / V1",\n      "changed": "AutoML-only LightGBM",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 3.163,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V1",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": 0.297\n    },\n    {\n      "experiment": "ADDENDUM / V2",\n      "changed": "Frozen representation plus AutoML logistic head",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 3.064,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V2",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": 0.198\n    },\n    {\n      "experiment": "ADDENDUM / V2c",\n      "changed": "Frozen representation logistic head",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 2.669,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V2c",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": -0.197\n    },\n    {\n      "experiment": "ADDENDUM / V3",\n      "changed": "Frozen representation plus AutoML LightGBM head",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 2.669,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V3",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": -0.197\n    },\n    {\n      "experiment": "ADDENDUM / V3c",\n      "changed": "Frozen representation LightGBM head",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 2.817,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V3c",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": -0.049\n    },\n    {\n      "experiment": "ADDENDUM / V4",\n      "changed": "Warm-start Transformer plus static AutoML branch",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 3.113,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V4",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": 0.247\n    },\n    {\n      "experiment": "ADDENDUM / V5",\n      "changed": "Warm-start zero-static control",\n      "population": "3,414 snapshots / 202 positive snapshots / retrospective TEST",\n      "lift10": 2.669,\n      "source": "experiments/automl_sequence/HISTORICAL_ADDENDUM_FINDINGS.md V5",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Point difference versus V0; combining inputs did not consistently increase lift",\n      "limitation": "Earlier upstream AutoML preparation used labels across splits; repeated TEST; no lift interval in original addendum",\n      "comparison_block": "R20261009A_FUSION_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "3 displayed decimals",\n      "delta10": -0.197\n    },\n    {\n      "experiment": "RECOVERED_TEST / Experimental AutoML LightGBM",\n      "changed": "Recovered scores on matched historical keys",\n      "population": "3,414 snapshots / 202 positives / retrospective TEST",\n      "lift10": 3.163,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Reference for current temporal comparison",\n      "limitation": "Reported recovered-artifact result; local predictions unavailable; upstream historical leakage and repeated TEST remain",\n      "comparison_block": "RECOVERED_MATCHED_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "Lift 3 decimals; delta supplied before rounding",\n      "delta10": 0,\n      "lower": null,\n      "upper": null\n    },\n    {\n      "experiment": "RECOVERED_TEST / Current temporal fusion",\n      "changed": "Recovered scores on matched historical keys",\n      "population": "3,414 snapshots / 202 positives / retrospective TEST",\n      "lift10": 3.113,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Ordering unresolved; interval crosses zero (not equivalence)",\n      "limitation": "Reported recovered-artifact result; local predictions unavailable; upstream historical leakage and repeated TEST remain",\n      "comparison_block": "RECOVERED_MATCHED_RETROSPECTIVE_TEST",\n      "split": "TEST_RETROSPECTIVE",\n      "precision": "Lift 3 decimals; delta supplied before rounding",\n      "delta10": -0.049,\n      "lower": -0.645,\n      "upper": 0.599\n    },\n    {\n      "experiment": "BLEND_A / 49-feature TRAIN-only refit",\n      "changed": "49 ordered encoded inputs; different fitting membership",\n      "population": "3,481 snapshots / 202 positives / VALIDATION",\n      "lift10": 2.52,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Descriptive comparison; this does not establish the cause of the gap",\n      "limitation": "Historical feature selection and production training membership unverified; not the 2.568 static control",\n      "comparison_block": "BLEND_A_49_AND_PRODUCTION_VALIDATION_DESCRIPTIVE",\n      "split": "VALIDATION_DESCRIPTIVE",\n      "precision": "Refit 2 decimals; production 3 decimals"\n    },\n    {\n      "experiment": "BLEND_A / Frozen production V63",\n      "changed": "49 ordered encoded inputs; different fitting membership",\n      "population": "3,481 snapshots / 202 positives / VALIDATION",\n      "lift10": 4.592,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Descriptive comparison; this does not establish the cause of the gap",\n      "limitation": "Historical feature selection and production training membership unverified; not the 2.568 static control",\n      "comparison_block": "BLEND_A_49_AND_PRODUCTION_VALIDATION_DESCRIPTIVE",\n      "split": "VALIDATION_DESCRIPTIVE",\n      "precision": "Refit 2 decimals; production 3 decimals"\n    },\n    {\n      "experiment": "Sleep-study and diagnosis interaction",\n      "changed": "Recent study in months 0-5 AND sleep-wake diagnosis in months 0-2",\n      "population": "16,256 TRAIN snapshots / 941 positives; 763 flagged",\n      "lift10": 2.07,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "REPORTED_NOT_LOCALLY_REPRODUCED",\n      "conclusion": "Flagged subgroup lift 3.10; feature ranking Lift@10% 2.07. These are different statistics.",\n      "limitation": "Conceived using TRAIN labels; top-decile ranking includes tied zero-score rows; incremental held-out benefit unmeasured",\n      "split": "TRAIN",\n      "precision": "2 displayed decimals"\n    },\n    {\n      "experiment": "Expanded time-aware feature matrix",\n      "changed": "49 V63 inputs plus 268 temporal summaries and three interactions",\n      "population": "23,151 original snapshots x 320 columns",\n      "lift10": null,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "FEATURES_REPORTED_MODEL_GAIN_PENDING",\n      "conclusion": "Timing and interaction features prepared; no model improvement inferred",\n      "limitation": "This feature expansion is on original keys, not the expanded patient cohort"\n    },\n    {\n      "experiment": "Expanded population / summary and NLP",\n      "changed": "48-month bounded cohort with inherited and hashed patient assignments",\n      "population": "449,199 snapshots / 25,925 patients / 7,486 positives; validation 67,969 / 1,141 positives",\n      "lift10": null,\n      "source": "experiments/challenger/EVIDENCE.md",\n      "evidence": "DOCUMENTED_SCREENSHOT_REPORTED",\n      "status": "AUDIT_REPORTED_RESULTS_PENDING",\n      "conclusion": "Population audit available; model lift remains pending",\n      "limitation": "Patient-disjoint development split; historical TEST exposure and source-vintage limitations remain"\n    },\n    {\n      "experiment": "CACHED_PILOT / AUTOML",\n      "changed": "Fixed 300-tree LightGBM scouting comparison; seed 42; two threads",\n      "population": "Complete cached TRAIN and fixed VALIDATION patients only",\n      "lift10": null,\n      "source": "experiments/cached_pilot/analysis.py",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_NOT_RUN",\n      "conclusion": "Await a checked report from the cached pilot",\n      "limitation": "Convenience population; no production comparison; NLP arms require matching validated artifacts"\n    },\n    {\n      "experiment": "CACHED_PILOT / AUTOML_TIME",\n      "changed": "Fixed 300-tree LightGBM scouting comparison; seed 42; two threads",\n      "population": "Complete cached TRAIN and fixed VALIDATION patients only",\n      "lift10": null,\n      "source": "experiments/cached_pilot/analysis.py",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_NOT_RUN",\n      "conclusion": "Await a checked report from the cached pilot",\n      "limitation": "Convenience population; no production comparison; NLP arms require matching validated artifacts"\n    },\n    {\n      "experiment": "CACHED_PILOT / AUTOML_NLP",\n      "changed": "Fixed 300-tree LightGBM scouting comparison; seed 42; two threads",\n      "population": "Complete cached TRAIN and fixed VALIDATION patients only",\n      "lift10": null,\n      "source": "experiments/cached_pilot/analysis.py",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_NOT_RUN",\n      "conclusion": "Await a checked report from the cached pilot",\n      "limitation": "Convenience population; no production comparison; NLP arms require matching validated artifacts"\n    },\n    {\n      "experiment": "CACHED_PILOT / AUTOML_TIME_NLP",\n      "changed": "Fixed 300-tree LightGBM scouting comparison; seed 42; two threads",\n      "population": "Complete cached TRAIN and fixed VALIDATION patients only",\n      "lift10": null,\n      "source": "experiments/cached_pilot/analysis.py",\n      "evidence": "SOURCE_CONFIGURATION_ONLY",\n      "status": "IMPLEMENTED_NOT_RUN",\n      "conclusion": "Await a checked report from the cached pilot",\n      "limitation": "Convenience population; no production comparison; NLP arms require matching validated artifacts"\n    }\n  ],\n  "engineering": [\n    {\n      "Change": "Set-based count materialization",\n      "Effect": "Snowflake table creation reported around 1 minute 56 seconds",\n      "Evidence": "EXECUTION_SCREENSHOT_REPORTED",\n      "Limitation": "Replaced repeated joins; tensor transfer and assembly still required. The earlier 39-hour projection was not a measured completed run.",\n      "Predictive impact": "Not measured; semantic parity must be checked"\n    },\n    {\n      "Change": "Sparse count assembly checkpoints",\n      "Effect": "2,304 saved snapshots matched staged values; 181 read blocks reported",\n      "Evidence": "EXECUTION_SCREENSHOT_REPORTED",\n      "Limitation": "Databricks transfer, conversion and compression still dominated around 79-85 seconds per reported block",\n      "Predictive impact": "None expected after exact parity; no model result inferred"\n    },\n    {\n      "Change": "Committed AutoML chunk reuse",\n      "Effect": "Verified completed chunks can resume without source requery",\n      "Evidence": "EXECUTION_SCREENSHOT_REPORTED",\n      "Limitation": "Chunk counts describe request batches, not model features or complete-cohort coverage",\n      "Predictive impact": "Not measured"\n    },\n    {\n      "Change": "Bounded cache pilot and aggregate tracker",\n      "Effect": "Independent checkpointed analysis; aggregate-only reporting",\n      "Evidence": "SOURCE_AND_SYNTHETIC_VERIFICATION",\n      "Limitation": "Clinical runtime and predictive results await execution",\n      "Predictive impact": "Pending"\n    }\n  ]\n}\n'
EXPECTED_HASHES = {'tracker': 'bb10c780ace5f727e946f256168807db5ce35652c50e8596cb2835541fbec112', 'evidence': '1cbb6a17207cd9c1d084bef6b44661dcac8190f8411350142869c767ddd8e07c', 'decoder': '65901cb2aaf5b49793ed597904872963ff4451e00aaf9ff4fbcf32f54a38b52b'}
for name, text in [("tracker", TRACKER_SOURCE), ("decoder", DECODER_SOURCE), ("evidence", EVIDENCE_SOURCE)]:
    if hashlib.sha256(text.encode()).hexdigest() != EXPECTED_HASHES[name]:
        raise ValueError("Embedded component checksum mismatch: " + name)
tracker = types.ModuleType("aggregate_evidence_tracker")
exec(compile(TRACKER_SOURCE, "aggregate_evidence_tracker", "exec"), tracker.__dict__)
exec(compile(DECODER_SOURCE, "checked_aggregate_decoder", "exec"), globals())
EVIDENCE = json.loads(EVIDENCE_SOURCE)
print("Embedded aggregate reader and evidence checksums verified.")


In [ ]:
reports, ingest_errors = [], []
if len(LOCAL_REPORT_PATHS) + len(REMOTE_REPORT_TABLES) > MAX_REPORTS:
    raise ValueError("Too many explicit reports for this bounded review")
for location in LOCAL_REPORT_PATHS:
    try:
        reports.append(tracker.load_local_report(location))
    except Exception as exc:
        ingest_errors.append({"Source": str(location), "Status": "BLOCKED", "Reason": str(exc)})
if REMOTE_REPORT_TABLES:
    if not all(sf_options.values()):
        raise ValueError("Fill the connection settings before requesting remote aggregate reports")
    if "spark" not in globals():
        raise RuntimeError("The remote aggregate reader requires the existing Spark session")
    def read_aggregate_query(query, maximum_rows):
        frame = spark.read.format("snowflake").options(**sf_options).option("query", query).load().limit(maximum_rows + 1).toPandas()
        frame.columns = [str(c).upper() for c in frame.columns]
        if len(frame) > maximum_rows:
            raise ValueError("Aggregate query exceeded its row bound")
        return frame
    for table in REMOTE_REPORT_TABLES:
        try:
            reports.append(tracker.load_remote_report(read_aggregate_query, table, decode_checked_artifact))
        except Exception as exc:
            ingest_errors.append({"Source": str(table), "Status": "BLOCKED", "Reason": str(exc)})
print("Checked aggregate reports:", len(reports))


In [ ]:
evidence_table, engineering_table, parse_errors = tracker.ledger(EVIDENCE, reports)
blocked_table = pd.concat([pd.DataFrame(ingest_errors), parse_errors], ignore_index=True)
display(evidence_table[tracker.DISPLAY_COLUMNS])
print("Feature subgroup association and model Lift@10% are different measurements.")
print("A verified file checksum establishes integrity, not an unbiased evaluation.")
if not blocked_table.empty:
    display(blocked_table)
print("Implemented or pending experiments:")
display(evidence_table[evidence_table["Lift@10%"].isna()][tracker.DISPLAY_COLUMNS])
print("Engineering changes are listed separately from predictive improvements:")
display(engineering_table)


In [ ]:
import matplotlib.pyplot as plt
for figure in tracker.plot_matched(evidence_table):
    display(figure)
    plt.close(figure)
progress = tracker.plot_progress(evidence_table)
if not progress:
    print("No refresh series with identical validation keys, labels and feature definitions is available yet.")
for figure in progress:
    display(figure)
    plt.close(figure)
print("Comparisons are grouped by recorded population, split and metric. No overall ranking is produced.")


In [ ]:
if OUTPUT_DIRECTORY:
    destination = Path(OUTPUT_DIRECTORY)
    destination.mkdir(parents=True, exist_ok=True)
    evidence_table.to_csv(destination / "experiment_evidence.csv", index=False)
    engineering_table.to_csv(destination / "engineering_evidence.csv", index=False)
    blocked_table.to_csv(destination / "blocked_reports.csv", index=False)
    payload = {"schema": "experiment_tracker_export_v1", "embedded_hashes": EXPECTED_HASHES,
               "report_receipts": [receipt for _, receipt in reports],
               "experiments": json.loads(evidence_table.to_json(orient="records")),
               "engineering": json.loads(engineering_table.to_json(orient="records"))}
    blob = json.dumps(payload, sort_keys=True, allow_nan=False, indent=2).encode()
    (destination / "tracker.json").write_bytes(blob)
    (destination / "tracker_receipt.json").write_text(json.dumps({"sha256": hashlib.sha256(blob).hexdigest()}), encoding="utf-8")
    print("Aggregate tracker saved:", str(destination))
else:
    print("Display complete. Set OUTPUT_DIRECTORY to save this aggregate ledger.")
